# L5c: Introduction to Linear Programming

How should we allocate limited resources among competing uses? We might decide how much of each product to purchase with a fixed budget, or how to send a required amount of flow through a network at minimum cost. Linear programming represents these decisions using a linear objective function, linear constraints, and bounds on continuous decision variables. The constraints describe which decisions are feasible; the objective allows us to compare them.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate resource-allocation models:__ Identify the decision variables, objective coefficients, constraints, and bounds in consumer-choice and minimum-cost-flow problems. Explain how each part of the linear program represents the original allocation question.
> * __Explain duality and resource values:__ Construct the dual of a resource-allocation model and explain how feasible dual solutions bound the primal objective. Use the consumer's budget constraint to interpret the optimal dual variable as the utility gained from an additional unit of budget.
> * __Evaluate optimization results:__ Distinguish a feasible decision from an optimal one. Check solver status, recompute the objective, and verify the constraints and bounds; explain how agreement between feasible primal and dual objective values certifies optimality.

We begin with the consumer's allocation problem, work through the choice between apples and oranges, and formulate network flow using an incidence matrix. We then return to the consumer model to develop its dual and interpret the value of the budget. These examples connect the mathematical formulation with the checks needed to interpret a solver's result.

Let's get started!

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the packages used for the allocation example.

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5800 course package documentation](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/) for the functions and types used in the companion example.

___

## Examples

We will use the following example to connect budget allocation with the geometry of a linear program:

> [▶ Apples, Oranges, and Linear Allocation](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). How does utility per dollar determine which fruit we should purchase? We hold prices and the budget fixed and solve three cases: apples offer more utility per dollar, oranges offer more, and both offer the same. We compare the quantities purchased, expenditure, and attained utility, then check the budget constraint and explain why equal ratios allow multiple optimal allocations.

___


## Primal Linear Programming Problems

Suppose we must choose how much of several activities to carry out while respecting limits on the resources they consume. In a __primal__ model, the decision variables describe the activities themselves: quantities purchased, production levels, or flows sent along network edges. We specify how these decisions contribute to the objective and how they use the available resources.

Let $n$ be the number of activities and $m$ the number of resource constraints. The vector $\mathbf{x}\in\mathbb{R}^{n}$ contains the activity levels, and the entry $c_i$ of the objective-coefficient vector $\mathbf{c}\in\mathbb{R}^{n}$ is the contribution per unit of activity $i$. The constraint matrix $\mathbf{A}\in\mathbb{R}^{m\times n}$ has one row per resource and one column per activity: $A_{j,i}$ is the amount of resource $j$ used per unit of activity $i$. The entry $b_j$ of the vector $\mathbf{b}\in\mathbb{R}^{m}$ gives the available amount of that resource.

> __Primal resource-allocation model:__
>
> When the activity levels are nonnegative and we want to maximize their total contribution, the linear program is given by:
> $$
> \begin{aligned}
> \underset{\mathbf{x}}{\text{maximize}}\quad
>     & O(\mathbf{x})=\mathbf{c}^{\top}\mathbf{x}
>       =\sum_{i=1}^{n}c_i x_i \\
> \text{subject to}\quad
>     & \mathbf{A}\mathbf{x}\leq\mathbf{b},\\
>     & \mathbf{x}\geq\mathbf{0}.
> \end{aligned}
> $$
> The vector inequalities apply entry by entry. Each resource constraint limits the combined use of that resource across all activities, while the nonnegativity bounds exclude negative activity levels.

For example, row $j$ of the matrix constraint expresses the following resource balance:
$$
\sum_{i=1}^{n}A_{j,i}x_i\leq b_j,
\qquad j=1,\ldots,m.
$$
Each term on the left has units of resource $j$, matching the units of $b_j$. The model is linear because each coefficient is fixed and the contributions from the activities are added together.

__What does it mean to solve this model?__ A decision vector is __feasible__ if it satisfies every constraint and bound. The collection of all feasible vectors is the __feasible region__. An optimal vector $\mathbf{x}^{\star}$ is feasible and attains the largest objective value over that region. A feasible vector need not be optimal, and several different vectors can attain the same optimal value.

If no feasible vector exists, the model is __infeasible__. If feasible vectors can produce arbitrarily large objective values, this maximization problem is __unbounded__ and has no finite optimum. These outcomes depend on the objective, constraints, and bounds together.

Linear programs can also minimize an objective, impose equality constraints, or use other lower and upper bounds. We will use the maximization form for consumer choice and an equality-constrained minimization form for network flow.

### Consumer Choice Problems as Linear Programs

Suppose we have a budget to spend on $n$ products and want to obtain the greatest total utility, where utility measures the satisfaction associated with consumption. Let $x_i\geq0$ be the quantity of product $i$ purchased, $p_i>0$ its price in dollars per unit, and $u_i\geq0$ its utility per unit. The available budget is $I\geq0$ dollars.

We treat quantities as continuous, so fractional purchases are allowed. We also assume that each additional unit contributes the same utility and that product availability does not impose another limit. Under these assumptions, the consumer-choice linear program is given by:
$$
\begin{aligned}
\underset{\mathbf{x}}{\text{maximize}}\quad
    & U(\mathbf{x})=\sum_{i=1}^{n}u_i x_i \\
\text{subject to}\quad
    & \sum_{i=1}^{n}p_i x_i\leq I,\\
    & x_i\geq0,\qquad i=1,\ldots,n.
\end{aligned}
$$
This is the resource-allocation model with one resource: the budget. The objective coefficients are the utilities, $\mathbf{c}=\mathbf{u}$, and the single row of the constraint matrix is $\mathbf{A}=[p_1\ \cdots\ p_n]$, with right-hand side $\mathbf{b}=[I]$.

### Worked Allocation: Apples and Oranges

Let's use the first case from the [allocation example](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). Apples cost 2 dollars per unit and contribute 0.55 utility units per unit; oranges cost 4 dollars per unit and contribute 0.45 utility units per unit. With a budget of 100 dollars, we obtain the following model:
$$
\begin{aligned}
\underset{x_A,x_O}{\text{maximize}}\quad
    & U(x_A,x_O)=0.55x_A+0.45x_O \\
\text{subject to}\quad
    & 2x_A+4x_O\leq100,\\
    & x_A,x_O\geq0,
\end{aligned}
$$
where the subscripts $A$ and $O$ denote apples and oranges. We can buy at most 50 units of apples or 25 units of oranges. Mixtures are also feasible as long as their total expenditure does not exceed the budget.

__Why would an optimal allocation use the entire budget?__ Both products have positive utility, quantities are divisible, and there are no stock limits. Any unspent budget could therefore purchase more fruit and increase total utility. At an optimum, the budget constraint must hold with equality. Solving that equality for the orange quantity gives:
$$
x_O=25-\frac{1}{2}x_A,
\qquad 0\leq x_A\leq50.
$$
Substituting this expression into the objective allows us to compare all allocations that use the full budget:
$$
\begin{aligned}
U(x_A)
&=0.55x_A+0.45\left(25-\frac{1}{2}x_A\right)\\
&=11.25+0.325x_A.
\end{aligned}
$$
The coefficient of $x_A$ is positive, so utility increases as we replace oranges with apples along the budget boundary. The largest feasible apple quantity is $x_A^{\star}=50$, which gives $x_O^{\star}=0$ and $U^{\star}=27.5$ utility units. Buying only oranges is also feasible and uses the full budget, but gives only 11.25 utility units. Using all available resources does not, by itself, establish optimality.

The same preference follows by comparing __utility per dollar__, $u_i/p_i$. For these products, the ratios are given by:
$$
\frac{u_A}{p_A}=\frac{0.55}{2}=0.275,
\qquad
\frac{u_O}{p_O}=\frac{0.45}{4}=0.1125.
$$
Each dollar spent on apples contributes more utility than a dollar spent on oranges. This comparison explains the optimal allocation for the present one-budget model and will help us interpret the dual variable later. The numerical result applies to these prices, utilities, and budget; changing them can change the preferred allocation.
### Comparing Allocations and Their Geometry

What changes if the consumer values the two fruits differently? We keep the prices at $p_A=2$ and $p_O=4$ dollars per unit and the budget at $I=100$ dollars, then change the utility coefficients. The following table compares the three cases in the [allocation example](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). The ratios measure utility per dollar, and the final column reports the maximum total utility:

<table style="margin: 12px auto; border-collapse: collapse; font-size: 95%;">
<thead>
<tr style="border-bottom: 1px solid #b8b8b8;">
<th style="padding: 5px 10px; text-align: left;">Case</th>
<th style="padding: 5px 10px; text-align: left;">Utilities<br>(<i>u</i><sub>A</sub>, <i>u</i><sub>O</sub>)</th>
<th style="padding: 5px 10px; text-align: right;">Apples<br><i>u</i><sub>A</sub>/<i>p</i><sub>A</sub></th>
<th style="padding: 5px 10px; text-align: right;">Oranges<br><i>u</i><sub>O</sub>/<i>p</i><sub>O</sub></th>
<th style="padding: 5px 10px; text-align: left;">Optimal allocation<br>(<i>x</i><sub>A</sub>, <i>x</i><sub>O</sub>)</th>
<th style="padding: 5px 10px; text-align: right;">Maximum<br>utility</th>
</tr>
</thead>
<tbody>
<tr><td style="padding: 5px 10px; text-align: left;">A</td><td style="padding: 5px 10px; text-align: left; white-space: nowrap;">(0.55, 0.45)</td><td style="padding: 5px 10px; text-align: right;">0.275</td><td style="padding: 5px 10px; text-align: right;">0.1125</td><td style="padding: 5px 10px; text-align: left;">(50, 0)</td><td style="padding: 5px 10px; text-align: right;">27.5</td></tr>
<tr><td style="padding: 5px 10px; text-align: left;">B</td><td style="padding: 5px 10px; text-align: left; white-space: nowrap;">(0.15, 0.55)</td><td style="padding: 5px 10px; text-align: right;">0.075</td><td style="padding: 5px 10px; text-align: right;">0.1375</td><td style="padding: 5px 10px; text-align: left;">(0, 25)</td><td style="padding: 5px 10px; text-align: right;">13.75</td></tr>
<tr><td style="padding: 5px 10px; text-align: left;">C</td><td style="padding: 5px 10px; text-align: left; white-space: nowrap;">(2, 4)</td><td style="padding: 5px 10px; text-align: right;">1</td><td style="padding: 5px 10px; text-align: right;">1</td><td style="padding: 5px 10px; text-align: left;">Any full-budget mixture</td><td style="padding: 5px 10px; text-align: right;">100</td></tr>
</tbody>
</table>

In Cases A and B, moving a dollar from the fruit with lower utility per dollar to the other fruit increases total utility. The optimal allocation therefore spends the entire budget on the fruit with the larger ratio. In Case C, each dollar contributes one unit of utility regardless of which fruit we purchase. Every nonnegative allocation on the budget boundary $2x_A+4x_O=100$ is optimal: buying 50 apples, buying 25 oranges, or buying 20 apples and 15 oranges all gives 100 utility units. The optimal value is unique even though the optimal allocation is not. These numerical results apply to the stated prices, budget, and utility coefficients.

We can also explain these outcomes using the geometry of the feasible region. Place apples on the horizontal axis and oranges on the vertical axis. Let $m_I$ denote the slope of the budget boundary, $m_O$ the slope of a line of constant objective value, and $\bar U$ a chosen total utility. With positive prices and utilities, solving the budget and utility equations for the orange quantity gives:
$$
\begin{aligned}
\text{Budget boundary:}\qquad
x_O &= \frac{I}{p_O}
       +\underbrace{\left(-\frac{p_A}{p_O}\right)}_{m_I}x_A,\\[6pt]
\text{Constant utility:}\qquad
x_O &= \frac{\bar U}{u_O}
       +\underbrace{\left(-\frac{u_A}{u_O}\right)}_{m_O}x_A.
\end{aligned}
$$
Increasing $\bar U$ shifts the constant-utility line upward without changing its slope. We seek the largest value of $\bar U$ whose line still intersects the feasible region. The following schematic shows where this last contact occurs for the three slope comparisons:

<div>
    <center>
        <img src="figs/Fig-ThreeCases-LP-Schematic.svg" width="1000" alt="Three schematic allocation cases: an optimal apple corner, an optimal orange corner, and an entire optimal budget edge when the utility-per-dollar ratios are equal."/>
    </center>
</div>

The gray triangle is the feasible region, the black diagonal is the budget boundary, and the colored diagonal lines represent different total utility values. The yellow markings identify the optimal corner or edge. The dashed outer box represents illustrative stock limits beyond the affordable quantities, so it does not restrict the shaded region. The figure shows the relative slopes schematically; the table supplies the numerical results.

* __Case A: $|m_O|>|m_I|$.__ The constant-utility lines are steeper than the budget boundary. Their last feasible contact is the apple corner, consistent with apples having the greater utility per dollar.
* __Case B: $|m_O|<|m_I|$.__ The constant-utility lines are flatter than the budget boundary. Their last feasible contact is the orange corner, consistent with oranges having the greater utility per dollar.
* __Case C: $|m_O|=|m_I|$.__ The lines are parallel to the budget boundary. At the largest feasible utility, one of these lines coincides with the entire budget edge, so both corners and every mixture between them are optimal.

This distinction matters when we interpret a solver's answer. For Case C, different solvers can return different quantities while agreeing on the maximum utility and satisfying the same budget constraint.
### Minimum Cost Network Flow Problems as Linear Programs
Another classic example of a resource allocation problem that can be formulated as a primal linear programming problem is the minimum cost maximum flow problem. In this problem, we have a directed (bipartite) graph with nodes representing sources, sinks, and intermediate nodes representing a matching process. The edges represent the flow of goods or resources between these nodes. Each edge has a capacity (the maximum amount of flow that can pass through it) and a cost per unit of flow.

> __Formulation__: Let the directed graph be represented as $G = (\mathcal{V}, \mathcal{E})$, where $\mathcal{V}$ is the set of vertices (nodes) and $\mathcal{E}$ is the set of edges. Each edge $j \in \mathcal{E}$ has a capacity $c_j$ and a cost (weight) $w_j$ per unit of flow. Let $f_j$ be the flow on edge $j$, and let $s$ be the source node and $t$ be the sink node. The goal is to maximize the flow from the source to the sink while minimizing the total cost of the flow.

We use an incidence matrix formulation where $\mathbf{A} \in \mathbb{R}^{|\mathcal{V}| \times |\mathcal{E}|}$ represents the graph structure. For node $i$ and edge $j$:
- $A_{ij} = 1$ if edge $j$ is incoming to node $i$
- $A_{ij} = -1$ if edge $j$ is outgoing from node $i$  
- $A_{ij} = 0$ otherwise

Putting all this together, we can formulate the minimum cost maximum flow problem as the _primal_ linear program:
$$
\begin{align*}
\text{minimize} &\, \sum_{j \in \mathcal{E}} w_j f_j \\
\text{subject to} \quad \mathbf{A}\mathbf{f} &= \mathbf{b}\\
~0 \leq f_j &\leq c_j \quad\forall j \in \mathcal{E}
\end{align*}
$$
where $\mathbf{f} \in \mathbb{R}^{|\mathcal{E}|}$ is the vector of flows on each edge, and $\mathbf{b} \in \mathbb{R}^{|\mathcal{V}|}$ is the right-hand side vector with:
$$
b_i = \begin{cases}
-F & \text{if } i = s \text{ (source generates flow)} \\
F & \text{if } i = t \text{ (sink consumes flow)} \\
0 & \text{otherwise (flow conservation)}
\end{cases}
$$
where $F$ is the total flow from the source to the sink. The optimal solution to this problem (if it exists) will give the flow on each edge that minimizes the total cost while satisfying the flow conservation constraints and capacity constraints.
___


## Dual Linear Programming Problems
Having defined primal linear programs, we now turn to their duals, alternative formulations that offer a different viewpoint on the same optimization. You can think of it as viewing the primal through a different lens.

If the _primal problem_ has the form:
$$
\begin{align*}
\text{maximize} &\, \sum_{i=1}^{n} c_{i}\;{x}_{i}\\
\text{subject to}~\sum_{i=1}^{n} A_{i,j}\;{x}_{i} &\leq b_{j}\quad j=1,2,\dots,m\\
~x_{i}&\geq {0}\qquad{i=1,2,\dots,n}
\end{align*}
$$
then the _dual problem_ has the form:
$$
\begin{aligned}
\text{minimize}\quad & \sum_{j=1}^{m} b_{j}\,y_{j}\\
\text{subject to}\quad & \sum_{j=1}^{m} A_{i,j}\,y_{j}\;\ge\;c_{i}
\quad&&i=1,2,\dots,n,\\
&y_{j}\;\ge\;0
\quad&&j=1,2,\dots,m.
\end{aligned}
$$

### What has changed?
There are several key differences between the primal and dual linear programming problems:
1. The objective function flips (maximum ⇒ minimum or minimum ⇒ maximum).
2. Primal objective coefficients $c_i$ become the dual right-hand side constants.
3. Primal right-hand side constants $b_j$ become the dual objective coefficients.
4. The $m\times n$ constraint matrix $A$ is transposed in the dual (so $A^\top$ appears).
5. The number of variables and constraints swap: the primal has $n$ variables, $m$ constraints, and the dual has $m$ variables and $n$ constraints.
6. Each primal constraint $a_j^\top x \le b_j$ gives a dual variable $y_j$. Each primal variable $x_i$ gives a dual constraint $(A^\top y)_i \ge c_i$.
7. Inequality directions and sign restrictions invert for the constraints: A $\le$ constraint in the primal gives rise to a $\ge$ constraint in the dual (and vice versa).
8. Equality constraints in the primal become free variables in the dual, i.e., $a_j^T x = b_j$ gives rise to a dual variable $y_j$ that is free (no sign restriction), while a dual constraint $A^\top y \ge c$ gives rise to a primal variable $x_i$ that is free.

Finally, the solutions of the primal and dual problems are related by the concept of __duality__. For a primal problem: $\max\{\,c^T x : A x \le b,\;x\ge0\}$ and its corresponding dual problem: $\min\{\,b^T y : A^T y \ge c,\;y\ge0\}$, the solutions are related:
* __Weak duality__: For any primal feasible $x$ and dual feasible $y$, we have $c^T x \le b^T y$. Thus, the primal optimum is always bounded above by the dual optimum. The difference between the two is called the _duality gap_.
* __Strong duality__: If both primal and dual are feasible and have finite optimal values, then $\max\{\,c^T x \} = \min\{\,b^T y\}$, i.e., the _duality gap is zero_. This means that the optimal values of the primal and dual problems are equal.

___


## How an LP solver fits the modeling workflow

The modeler supplies decision variables, a linear objective, linear constraints, and variable bounds. A solver then returns a status and a candidate solution. Our responsibility is to check the status, recompute the objective, and verify feasibility before interpreting the answer.

The supporting revised-simplex notebook opens one solver-internals window: basic variables select a corner, reduced costs identify a potentially improving direction, and a ratio test preserves feasibility. Interior-point derivations and duality proofs are deeper-dive material rather than required Week 5 content.


## Summary

- An LP separates decisions, objective coefficients, constraints, and bounds.
- The primal describes activities; the dual assigns marginal values to limiting resources.
- Network-flow conservation is an incidence-matrix equality, so minimum-cost flow is an LP.
- Solver status and an independent feasibility check are part of the result.
